# 05주 · 결합분포·공분산·독립성

공학통계학 CE202

**목표**: 자료·확률·분포·추정·검정의 원리 이해와 통계적 해석 심화

학습 주제: 결합확률분포 / 공분산

[Excel 실습](https://raw.githubusercontent.com/steelpl/lecture/main/CE202-statistics-for-engineers/week05/CE202_W05.xlsx) · [주차 안내](https://github.com/steelpl/lecture/tree/main/CE202-statistics-for-engineers/week05)

질문·가정 확인 → 손계산·예측 → 계산·시각화 → 한 조건 변경 → 통계적 해석·한계 설명

셀 실행: Shift+Enter. 아래 코드의 None은 직접 채울 항목. 기준 결과와 조건을 바꾼 결과를 비교

핵심 실습: 손계산·원리와 계산식의 연결·조건 변경·해석. AI 활용은 개념 학습 후 선택 활동

[공통 학습 계획](https://github.com/steelpl/lecture/blob/main/CE202-statistics-for-engineers/COURSE_PLAN.md) · [학기 운영안내](https://github.com/steelpl/lecture/blob/main/CURRENT_OFFERING.md)


## 0. 실행 환경

필요한 패키지의 최소 버전 확인. 기본 코드와 보조 함수는 이번 주 실습에 필요한 도구. 세부 구현 암기 불필요


In [ ]:
# 최초 실행 시 필요한 버전만 설치. Colab에서 위에서 아래로 실행
import importlib.metadata as metadata, subprocess, sys
from packaging.version import Version
requirements = {"numpy": "1.26", "pandas": "2.2", "scipy": "1.16", "matplotlib": "3.10", "openpyxl": "3.1"}
for package, minimum in requirements.items():
    try:
        outdated = Version(metadata.version(package)) < Version(minimum)
    except metadata.PackageNotFoundError:
        outdated = True
    if outdated:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", f"{package}>={minimum}"])
print("실행 환경 준비 완료. 이미 불러온 패키지를 업데이트한 경우 런타임 재시작 후 처음부터 실행")

In [ ]:
import math, tempfile, base64, zipfile, io
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
WORK = Path(tempfile.mkdtemp(prefix="ce202-"))
rng = np.random.default_rng(2027)

def vector(values, minimum=1):
    x = np.asarray(values, dtype=float)
    if x.ndim != 1 or len(x) < minimum or not np.isfinite(x).all():
        raise ValueError("유한한 1차원 수치 자료와 최소 표본수 확인 필요")
    return x

def probabilities(values):
    p = vector(values)
    if np.any(p < 0) or not np.isclose(p.sum(), 1):
        raise ValueError("확률의 비음수성·합계 1 확인 필요")
    return p

def mean_ci(values, confidence=0.95):
    x = vector(values, 2)
    if not 0 < confidence < 1:
        raise ValueError("신뢰수준은 0과 1 사이")
    se = x.std(ddof=1) / np.sqrt(len(x))
    q = stats.t.ppf((1 + confidence) / 2, df=len(x)-1)
    return {"n": len(x), "mean": x.mean(), "se": se,
            "df": len(x)-1, "low": x.mean()-q*se, "high": x.mean()+q*se}

def wilson_interval(successes, n, confidence=0.95):
    if not isinstance(n, (int, np.integer)) or n <= 0 or not 0 <= successes <= n:
        raise ValueError("0 ≤ 성공수 ≤ 양의 정수 표본수 확인 필요")
    if int(successes) != successes or not 0 < confidence < 1:
        raise ValueError("정수 성공수·신뢰수준 확인 필요")
    z = stats.norm.ppf((1+confidence)/2)
    phat = successes / n
    den = 1 + z*z/n
    center = (phat + z*z/(2*n)) / den
    half = z / den * np.sqrt(phat*(1-phat)/n + z*z/(4*n*n))
    return center-half, center+half

def anova_summary(groups):
    arrays = [vector(g, 2) for g in groups]
    if len(arrays) < 2:
        raise ValueError("두 집단 이상 필요")
    n = np.array([len(g) for g in arrays]); means = np.array([g.mean() for g in arrays])
    grand = np.concatenate(arrays).mean()
    ss_between = np.sum(n*(means-grand)**2)
    ss_within = sum(np.sum((g-g.mean())**2) for g in arrays)
    df_between = len(arrays)-1; df_within = n.sum()-len(arrays)
    if ss_within <= 0:
        raise ValueError("집단 내 변동이 0이면 일반적인 F 계산 불가")
    F = (ss_between/df_between) / (ss_within/df_within)
    return {"ss_between": ss_between, "ss_within": ss_within,
            "df_between": df_between, "df_within": df_within,
            "F": F, "p": stats.f.sf(F, df_between, df_within)}

def regression(x, y):
    x = vector(x, 3); y = vector(y, 3)
    if len(x) != len(y) or np.ptp(x) == 0:
        raise ValueError("같은 길이·서로 다른 x 값 확인 필요")
    fit = stats.linregress(x, y)
    fitted = fit.intercept + fit.slope*x
    residual = y-fitted
    sse = np.sum(residual**2); sst = np.sum((y-y.mean())**2)
    if sst <= 0:
        raise ValueError("y 변동이 0이면 일반적인 R² 해석 불가")
    return {"slope": fit.slope, "intercept": fit.intercept, "r2": 1-sse/sst,
            "sse": sse, "mse_residual": sse/(len(x)-2), "mse_prediction": sse/len(x),
            "p": fit.pvalue, "stderr": fit.stderr, "fitted": fitted, "residual": residual}


<a id="s01"></a>

## 공 주머니의 결합분포

**먼저 예측**: 비복원추출 2개에서 파랑 2·빨강 2가 가능한가?

계산 전에 분모·단위·조건을 한 줄로 기록


In [ ]:
rows=[]
for blue in range(3):
    for red in range(3):
        green=2-blue-red
        mass=math.comb(3,blue)*math.comb(2,red)*math.comb(3,green)/math.comb(8,2) if 0<=green<=3 else 0
        rows.append((blue,red,mass))
joint=pd.DataFrame(rows,columns=["blue","red","p"])
print(joint.pivot(index="blue",columns="red",values="p"))
assert np.isclose(joint.p.sum(),1)
print("주변분포:",joint.groupby("blue").p.sum())

**조건 변경**: 복원추출로 바꾼 사본과 가능한 사건·확률 비교

위 코드를 복사한 셀에서 한 조건씩 변경. 기준 셀은 보존

**선택 심화 · AI 검토**: 파랑3·빨강2·초록3, 동시 비복원추출2를 명시하고 결합·주변확률 요청

**결과 검토**: 불가능한 조합의 확률 0과 정규화 확인

예측·손계산·변경 결과·개념 설명·적용 한계 기록


In [ ]:
# 변경한 코드·확인할 식을 이 셀에 추가


<a id="s02"></a>

## 결합분포에서 공분산 계산

**먼저 예측**: 총 추출수가 고정되면 두 색 개수의 관계는?

계산 전에 분모·단위·조건을 한 줄로 기록


In [ ]:
# 이번 차시의 원본 결합자료 설정. 공통 준비 셀 실행 후 이 절부터 시작 가능
rows=[]
for blue in range(3):
    for red in range(3):
        green=2-blue-red
        mass=math.comb(3,blue)*math.comb(2,red)*math.comb(3,green)/math.comb(8,2) if 0<=green<=3 else 0
        rows.append((blue,red,mass))
joint=pd.DataFrame(rows,columns=["blue","red","p"])
eb=np.dot(joint.blue,joint.p); er=np.dot(joint.red,joint.p)
cov=np.dot((joint.blue-eb)*(joint.red-er),joint.p)
print({"E(blue)":eb,"E(red)":er,"Cov":cov})
assert np.isclose(cov,-9/56)
assert not np.isclose(joint.query("blue==2 and red==2").p.iloc[0], joint.query("blue==2").p.sum()*joint.query("red==2").p.sum())

**조건 변경**: 서로 독립인 두 동전 결합표와 비교

위 코드를 복사한 셀에서 한 조건씩 변경. 기준 셀은 보존

**선택 심화 · AI 검토**: 독립성은 모든 결합확률과 주변확률 곱으로 확인하도록 요청

**결과 검토**: 공분산·상관계수의 단위와 선형관계 의미 구분

예측·손계산·변경 결과·개념 설명·적용 한계 기록


In [ ]:
# 변경한 코드·확인할 식을 이 셀에 추가


<a id="s03"></a>

## 공분산 0의 반례

**먼저 예측**: 공분산 0이 독립을 보장하는가?

계산 전에 분모·단위·조건을 한 줄로 기록


In [ ]:
x=np.array([-1.,0,1]); y=x*x; p=np.repeat(1/3,3)
cov0=np.dot((x-np.dot(x,p))*(y-np.dot(y,p)),p)
print("공분산:",cov0,"X=0이면 Y=0으로 결정")
assert np.isclose(cov0,0) and y[1]==0 and np.mean(y==0)==1/3

**조건 변경**: x 범위와 확률을 바꿔 공분산·결정 관계 관찰

위 코드를 복사한 셀에서 한 조건씩 변경. 기준 셀은 보존

**선택 심화 · AI 검토**: 무상관과 독립의 역명제가 성립하지 않는 반례 설명 요청

**결과 검토**: 검토용 추가 예시. 원본의 관측자료로 표시하지 않음

예측·손계산·변경 결과·개념 설명·적용 한계 기록


In [ ]:
# 변경한 코드·확인할 식을 이 셀에 추가


## 주차 확인

결합확률 합계와 공분산 해석 기록

입력 항목:

- `joint_probability_sum`: 원본 결합확률표의 전체 확률 합계 · 수치
- `covariance`: 파랑·빨강 공 개수의 공분산 · 수치
- `zero_covariance_implies_independence`: 공분산 0이면 항상 독립인가? · Python `True`/`False`

공개 확인용 항목. 결과값만 복사한 경우 학습 완료로 판단 불가. 분석 가정·자료 출처·추론 범위는 아래 기록과 함께 검토


In [ ]:
answers = {'joint_probability_sum': None, 'covariance': None, 'zero_covariance_implies_independence': None}


In [ ]:
def check_answers(values):
    expected = {'joint_probability_sum': 1.0, 'covariance': -0.16071428571428573, 'zero_covariance_implies_independence': False}
    pending, wrong = [], []
    for key, target in expected.items():
        got = values.get(key)
        if got is None:
            pending.append(key)
        elif isinstance(target, bool):
            if not isinstance(got, (bool, np.bool_)) or bool(got) != target:
                wrong.append(key)
        elif isinstance(target, str):
            if not isinstance(got, str) or got.strip().lower() != target:
                wrong.append(key)
        elif isinstance(got, (bool, np.bool_)) or not isinstance(got, (int, float, np.number)) or not np.isfinite(got) or not np.isclose(got, target, rtol=1e-6, atol=1e-8):
            wrong.append(key)
    if pending:
        print("미완료:", ", ".join(pending))
    if wrong:
        print("재검토:", ", ".join(wrong))
    if not pending and not wrong:
        print("공개 계산·개념 확인 통과. 해석·가정·AI 검토 기록은 별도 확인 필요")
    return {"pending": pending, "wrong": wrong, "passed": not pending and not wrong}

check_result = check_answers(answers)

## 원리 탐구·결과 해석 기록

1. 문제·자료·단위·모형과 가정
2. 실행 전 예측·작은 손계산과 근거
3. 기준 계산의 중간값·결과와 원리의 연결
4. 한 조건 변경 결과와 차이의 원인
5. 독립 검산·적용 조건·불확실성·한계
6. 자신의 말로 설명할 핵심 원리 한 가지

AI 활용은 선택. 사용 시 요청·의심한 출력·수정·검산 근거를 별도 기록. 글자 수 자동 채점 없음


In [ ]:
review_log = {
    "문제·자료·단위·가정": None,
    "예측·손계산 근거": None,
    "중간 계산·원리 연결": None,
    "조건 변경·차이의 원인": None,
    "검산·해석·적용 한계": None,
    "내 설명": None,
}
# 선택 활동: AI 활용 시 요청·의심한 출력·수정·검산 근거를 별도 기록
